# Qlib Research Guide (Google Colab Friendly)

This notebook is a kid-friendly ("explain like I'm 5") tour of [Qlib](https://github.com/microsoft/qlib). It walks through installing Qlib, loading data, building models, forecasting, backtesting, and analyzing performance. Every code cell is meant to run top-to-bottom in Google Colab.


## Part 1 — What is Qlib?

* **Qlib** is like a big box of Lego blocks for stock market research. You pick the pieces (data, features, models), snap them together, and test trading ideas.
* People use it because it already knows how to fetch market data, clean it, train models, and run backtests, so you can focus on ideas instead of wiring.
* Important words (kid-style):
  * **Market data**: the prices and volumes of many stocks every day.
  * **Features (factors)**: numbers we make from prices (like a 5-day average). They are clues the model can learn from.
  * **Labels**: the answer we want the model to guess (future return).
  * **Model**: a smart calculator that learns patterns from features and labels.
  * **Alpha / Signal / Score**: the model’s guess of which stocks will go up. Higher score → we think it will rise more.
  * **Backtest**: a time machine that pretends we traded in the past to see if the idea works.


## Part 2 — Installation & Setup (Colab-ready)

Run these steps once at the top of Colab. They install Qlib, download sample data, and initialize the framework.


In [ ]:
# Install from GitHub to get the newest Qlib (works in Colab)
!pip -q install "git+https://github.com/microsoft/qlib.git#egg=pyqlib"

# (Optional but handy) Install plotting helpers
!pip -q install matplotlib seaborn


In [ ]:
# Imports used throughout the notebook
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import qlib
from qlib.constant import REG_CN, REG_US
from qlib.utils import init_instance_by_config
from qlib.data.dataset import DatasetH
from qlib.contrib.data.handler import DataHandlerLP
from qlib.contrib.model.gbdt import LGBModel
from qlib.contrib.model.linear import LinearModel
from qlib.contrib.model.pytorch_gru import GRUModel
from qlib.contrib.model.pytorch_gat import GATModel
from qlib.contrib.strategy import TopkDropoutStrategy
from qlib.contrib.executor import SimulatorExecutor
from qlib.contrib.evaluate import backtest, risk_analysis
from qlib.contrib.report import analysis_position


### Download sample data (China + US)
The helper script fetches daily bars. It may take a few minutes the first time.


In [ ]:
# Download CN & US data sets to the user directory (~/.qlib)
!python -m qlib.run.get_data --target_dir ~/.qlib/qlib_data/cn_data --region cn --download_mode auto
!python -m qlib.run.get_data --target_dir ~/.qlib/qlib_data/us_data --region us --download_mode auto


In [ ]:
# Choose which market to work with (CN by default)
MARKET = "csi300"        # China A-share large-cap index
BENCHMARK = "SH000300"    # Benchmark for backtests
DATA_PATH = os.path.expanduser("~/.qlib/qlib_data/cn_data")

# Initialize Qlib
qlib.init(provider_uri=DATA_PATH, region=REG_CN, expression_cache=None, dataset_cache=None)

print("Qlib ready! Version:", qlib.__version__)
print("Data path exists:", os.path.exists(DATA_PATH))


## Part 3 — Data Processing

Here we teach Qlib what data to use and how to clean it.

* **Dataset initialization**: tells Qlib which dates, which stocks, and what features/labels to prepare.
* **DataHandler**: a friendly kitchen robot that slices, cleans, and normalizes the data.
* **Processors**: tiny tools the robot uses, like filling missing values or ranking numbers.


In [ ]:
# Configuration for the data handler (feature engineering + labeling)
handler_config = {
    "class": "DataHandlerLP",
    "module_path": "qlib.contrib.data.handler",
    "kwargs": {
        "start_time": "2017-01-01",
        "end_time": "2022-12-31",
        "fit_start_time": "2017-01-01",
        "fit_end_time": "2020-12-31",
        "instruments": MARKET,
        # Features: simple price/volume + rolling stats
        "feature": [
            "$close",
            "$open",
            "$high",
            "$low",
            "$volume",
            "Ref($close, 1)",
            "Ref($close, 5)",
            "Mean($close, 5)",
            "Mean($close, 20)",
            "Std($close, 5)",
            "Std($close, 20)",
            "Mean($volume, 20)",
            "$close / Ref($close, 5) - 1",
        ],
        # Label: 2-day future return (tomorrow vs. the next day)
        "label": ["Ref($close, -2) / Ref($close, -1) - 1"],
        # Learn-time processors: drop bad labels, normalize features & labels per day
        "learn_processors": [
            {"class": "DropnaLabel"},
            {"class": "CSRankNorm", "kwargs": {"fields_group": "label"}},
            {"class": "RobustZScoreNorm", "kwargs": {"fields_group": "feature"}},
            {"class": "Fillna", "kwargs": {"fields_group": "feature"}},
        ],
        # Infer-time processors: same normalization for inference
        "infer_processors": [
            {"class": "RobustZScoreNorm", "kwargs": {"fields_group": "feature"}},
            {"class": "Fillna", "kwargs": {"fields_group": "feature"}},
        ],
    },
}

# Segment the calendar into train/valid/test
segments = {
    "train": ("2017-01-01", "2019-12-31"),
    "valid": ("2020-01-01", "2020-06-30"),
    "test": ("2020-07-01", "2022-12-31"),
}

# Build the dataset
dataset = DatasetH(handler=handler_config, segments=segments)

print(dataset)


### Peek at the raw data
We pull a small slice to see what the robot prepared. Columns are grouped as **feature** and **label**.


In [ ]:
# Load a tidy DataFrame for the training split
train_df = dataset.prepare("train")  # contains both features and label
train_df.columns = [f"{g}::{n}" for g, n in train_df.columns]
print(train_df.head())

# Plot one feature over time for a single stock
sample_instrument = train_df.index.get_level_values("instrument").unique()[0]
plot_df = train_df.xs(sample_instrument, level="instrument").reset_index()

plt.figure(figsize=(10, 4))
plt.plot(plot_df["datetime"], plot_df["feature::$close"], label="Close price")
plt.title(f"Sample close prices: {sample_instrument}")
plt.xlabel("Date")
plt.ylabel("Price")
plt.legend()
plt.tight_layout()
plt.show()


### Feature distributions
Checking feature shapes helps catch weird values.


In [ ]:
feature_cols = [c for c in train_df.columns if c.startswith("feature::")]
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.histplot(train_df[feature_cols[0]].sample(5000), bins=40, ax=axes[0])
axes[0].set_title("Feature distribution (example)")

sns.histplot(train_df[feature_cols[1]].sample(5000), bins=40, ax=axes[1])
axes[1].set_title("Another feature distribution")
plt.tight_layout()
plt.show()


## Part 4 — Modeling (Model Zoo)
We try several models and explain the settings like bedtime stories:

* **Linear model**: like drawing a straight line through the data.
* **LightGBM (LGBM)**: many tiny decision trees that vote together.
* **GRU (DNN)**: a memory-friendly neural net that remembers short histories.
* **GAT (Graph Attention)**: a neural net that lets stocks “talk” to each other.

Hyperparameters (kid-style):
* `learning_rate`: how big each learning step is.
* `num_leaves` / `hidden_size`: how big the model’s brain is.
* `n_estimators` / `layers`: how many times we let the model think.


In [ ]:
# Helper: train a model and get predictions on all splits

def train_and_predict(model, dataset, name):
    print(f"Training {name}...")
    model.fit(dataset)
    pred = model.predict(dataset)
    print(f"{name} done. Prediction sample:
", pred.head())
    return pred

# 1) Linear baseline
linear_model = LinearModel()
linear_pred = train_and_predict(linear_model, dataset, "LinearModel")

# 2) Gradient-boosted trees (LightGBM)
lgb_model = LGBModel(
    loss="mse",
    learning_rate=0.05,
    num_leaves=64,
    n_estimators=200,
    max_depth=-1,
    subsample=0.8,
    colsample_bytree=0.8,
)
lgb_pred = train_and_predict(lgb_model, dataset, "LGBModel")

# 3) GRU deep model
gru_model = GRUModel(
    d_feat=dataset.handler.get_feature_dim(),
    hidden_size=64,
    num_layers=2,
    drop_prob=0.1,
    n_epochs=10,
    lr=1e-3,
    batch_size=1024,
    early_stop=5,
    metric="loss",
    loss="mse",
)
gru_pred = train_and_predict(gru_model, dataset, "GRUModel")

# 4) Graph Attention Network (GAT)
gat_model = GATModel(
    d_feat=dataset.handler.get_feature_dim(),
    hidden_size=64,
    num_layers=2,
    n_epochs=10,
    lr=1e-3,
    batch_size=512,
    drop_prob=0.1,
    metric="loss",
    loss="mse",
)
gat_pred = train_and_predict(gat_model, dataset, "GATModel")


## Part 5 — Forecasting / Inference

We use the trained model to score the test split. Higher score = stronger belief the stock will rise.


In [ ]:
# Pick one model for the rest of the demo
best_pred = lgb_pred  # swap to linear_pred / gru_pred / gat_pred if desired

# Focus on the test period only
pred_test = best_pred.loc[pd.IndexSlice[segments["test"][0]:segments["test"][1], :]]
print(pred_test.head())

# Plot prediction distribution
plt.figure(figsize=(8,4))
sns.histplot(pred_test.values.flatten(), bins=50, kde=True)
plt.title("Prediction (alpha) distribution on test period")
plt.xlabel("Score")
plt.tight_layout()
plt.show()


Explain the jargon like a 5-year-old:

* **Alpha / Signal / Score**: the model’s grade for each stock day. Higher = the model thinks it will go up more soon.
* **Prediction**: turning today’s features into that score.
* **Save predictions**: helpful for later backtests or comparisons.


In [ ]:
# Save predictions to disk (optional)
pred_path = "predictions.parquet"
pred_test.to_frame("score").to_parquet(pred_path)
print("Saved to", pred_path)


## Part 6 — Portfolio Construction & Backtest

We turn scores into trades and simulate them:

* **Strategy**: pick the top-K scores and drop the worst each day.
* **Executor**: a simulator that applies costs and keeps positions.
* **Backtest**: walks through time to see profits, like a pretend time machine.
* Kid words for metrics:
  * **Sharpe ratio**: how much return per unit of bounciness (volatility). Bigger is better.
  * **Max drawdown**: the biggest peak-to-valley fall. Smaller is safer.
  * **Turnover**: how much we trade. High turnover means more buying/selling.


In [ ]:
# Build strategy from the prediction scores
strategy = TopkDropoutStrategy(signal=pred_test, topk=30, n_drop=5)
executor = SimulatorExecutor(time_per_step="day", generate_portfolio_metrics=True)

backtest_config = {
    "benchmark": BENCHMARK,
    "account": 100_000_000,
    "buy_cost": 0.0015,
    "sell_cost": 0.0015,
    "min_cost": 5,
}

report, positions = backtest(
    executor=executor,
    strategy=strategy,
    **backtest_config,
)

print("Backtest report head:
", report.head())


In [ ]:
# Plot equity curve from backtest
analysis_df = analysis_position.report_graphic(report, show=False)

plt.figure(figsize=(10,4))
plt.plot(report.index, report["return"] + 1.0, label="Strategy (cum)")
plt.title("Portfolio equity curve (simulation)")
plt.xlabel("Date")
plt.ylabel("Cumulative return")
plt.legend()
plt.tight_layout()
plt.show()


## Part 7 — Performance Analysis

We grade the model and the backtest:

* **IC (Information Coefficient)**: correlation between scores and future returns. Like checking if high scores really meant higher returns.
* **ICIR**: average IC divided by its wiggles. More stable is better.
* **Cumulative / annualized return**: how much money grew overall and per year.
* **Risk metrics**: volatility, max drawdown, turnover.
* **Feature importance**: which features mattered most (for tree models).


In [ ]:
# Compute risk analysis (includes IC & ICIR when signals and labels align)
analysis = risk_analysis(report, positions=positions, benchmark=BENCHMARK)
for k, v in analysis.items():
    print(f"{k}:
{v}
")


In [ ]:
# Feature importance for the LightGBM model
if hasattr(lgb_model, "model"):
    lgb_importance = pd.Series(lgb_model.model.feature_importances_, index=lgb_model.feature_names)
    lgb_importance = lgb_importance.sort_values(ascending=False).head(20)
    plt.figure(figsize=(8,6))
    sns.barplot(x=lgb_importance.values, y=lgb_importance.index)
    plt.title("Top feature importances (LGBM)")
    plt.xlabel("Importance")
    plt.ylabel("Feature")
    plt.tight_layout()
    plt.show()


In [ ]:
# Confusion-matrix-like check: bucket predictions and compare returns
label_df = dataset.prepare("test", col_set=["label"], data_key=DataHandlerLP.DK_L)
label_df.columns = [f"{g}::{n}" for g, n in label_df.columns]

pred_with_label = label_df.copy()
pred_with_label["score"] = pred_test
pred_with_label = pred_with_label.dropna()

pred_with_label["score_bucket"] = pd.qcut(pred_with_label["score"], 5, labels=False)

bucket_return = pred_with_label.groupby("score_bucket")["label::Ref($close, -2) / Ref($close, -1) - 1"].mean()
plt.figure(figsize=(8,4))
plt.bar(bucket_return.index.astype(str), bucket_return.values)
plt.title("Average future return by score bucket")
plt.xlabel("Score bucket (0=low, 4=high)")
plt.ylabel("Mean future return")
plt.tight_layout()
plt.show()


## Part 8 — Advanced Qlib Tricks

* **Online serving**: save a trained model and load it to serve daily signals.
* **Optimization**: tune hyperparameters with validation split.
* **Workflow pipelines**: chain data → model → backtest with configs.
* **Custom DataHandler**: subclass `DataHandlerLP` and override `feature` or processors.
* **Custom model**: implement `fit` and `predict` to plug into Qlib.


In [ ]:
# Save & reload a trained model (serving-style)
model_path = "lgb_model.pkl"
lgb_model.save(model_path)
print("Saved model to", model_path)

loaded_model = LGBModel()
loaded_model.load(model_path)
reloaded_pred = loaded_model.predict(dataset)
print("Loaded model works, sample:
", reloaded_pred.head())


In [ ]:
# Hyperparameter search example (very small for demo)
search_spaces = [
    {"learning_rate": 0.05, "num_leaves": 64},
    {"learning_rate": 0.1, "num_leaves": 128},
]

best_score, best_params = None, None
label_valid = dataset.prepare("valid", col_set=["label"], data_key=DataHandlerLP.DK_L)
label_valid.columns = [f"{g}::{n}" for g, n in label_valid.columns]
label_valid_series = label_valid.iloc[:, 0]

for params in search_spaces:
    trial_model = LGBModel(**params, n_estimators=100)
    trial_model.fit(dataset)
    pred = trial_model.predict(dataset)
    aligned = pred.loc[label_valid_series.index].squeeze()
    ic = aligned.corr(label_valid_series)
    print(f"Params {params} -> IC {ic:.4f}")
    if best_score is None or ic > best_score:
        best_score, best_params = ic, params

print("Best params:", best_params, "with IC", best_score)


In [ ]:
# Template for a custom DataHandler
from qlib.data.dataset.handler import DataHandlerLP as BaseHandler

class MyCustomHandler(BaseHandler):
    def __init__(self, **kwargs):
        super().__init__(**kwargs)
        # You can override feature/label definitions
        self._feature = ["$close", "Mean($close, 10)"]
        self._label = ["Ref($close, -2)/Ref($close, -1) - 1"]

    # Optional: override get_feature or other hooks

print("Custom handler ready to be plugged into DatasetH.")


In [ ]:
# Template for a custom model
from qlib.model.base import Model

class MySimpleModel(Model):
    def fit(self, dataset):
        df = dataset.prepare("train")
        df.columns = [f"{g}::{n}" for g, n in df.columns]
        X = df.filter(like="feature::").values
        y = df.filter(like="label::").values.ravel()
        from sklearn.linear_model import LinearRegression
        self.reg = LinearRegression().fit(X, y)
        self.feature_names = df.filter(like="feature::").columns
        return self

    def predict(self, dataset):
        df = dataset.prepare("test")
        df.columns = [f"{g}::{n}" for g, n in df.columns]
        X = df.filter(like="feature::").values
        pred = pd.Series(self.reg.predict(X), index=df.index)
        return pred

print("Custom model template defined.")


## Part 9 — Summary & Next Steps

* We built a full Qlib workflow: install → data → features → models → forecasts → backtest → analysis.
* Next adventures:
  * Try more features (e.g., Alpha360 handler) or longer history.
  * Experiment with the US market by switching `REG_US` and data path.
  * Tune hyperparameters with more candidates.
  * Deploy the saved model for daily signals.
* Helpful links:
  * [Qlib Documentation](https://qlib.readthedocs.io/)
  * [Qlib GitHub](https://github.com/microsoft/qlib)
  * [Data Guide](https://qlib.readthedocs.io/en/latest/component/data.html)
  * [Model Zoo](https://qlib.readthedocs.io/en/latest/component/model.html)
  * [Workflow examples](https://qlib.readthedocs.io/en/latest/component/workflow.html)
